# AFT-001: Naive Retry Amplification -- Demo**Class:** Loop & Recursion | **Severity:** P1Demonstrates how retry logic that appends (instead of replaces) tool resultscauses token blowup. Shows the wrong fix (more retries) and the correct fix(idempotency key with replace semantics).

In [ ]:
import hashlib, json, randomfrom dataclasses import dataclass, fieldfrom typing import Any@dataclassclass ToolResult:    call_id: str    tool_name: str    status: str    data: Any = None    error: str | None = None    def to_message(self) -> dict:        return {"role":"tool","call_id":self.call_id,"name":self.tool_name,                "content":json.dumps({"status":self.status,"data":self.data,"error":self.error})}@dataclassclass MessageHistory:    messages: list[dict] = field(default_factory=list)    def token_estimate(self) -> int:        return sum(len(json.dumps(m))//4 for m in self.messages)    def tool_count(self) -> int:        return sum(1 for m in self.messages if m.get("role")=="tool")

## The Failure: Naive Retry (Append Every Attempt)Each retry is appended as a separate tool result. The LLM sees timeout errorsas valid tool results and reasons over all of them.

In [ ]:
class NaiveRetryExecutor:    def __init__(self, history, max_retries=3):        self.history = history        self.max_retries = max_retries    def execute(self, tool_name, args, fail_rate=0.7):        for attempt in range(self.max_retries):            cid = f"{tool_name}_{attempt}_{hashlib.md5(json.dumps(args).encode()).hexdigest()[:8]}"            if random.random() < fail_rate and attempt < self.max_retries - 1:                r = ToolResult(cid, tool_name, "error", error=f"Timeout attempt {attempt+1}")            else:                r = ToolResult(cid, tool_name, "success", data={"customers":[{"id":1}]})            self.history.messages.append(r.to_message())  # BUG: append every attempt            if r.status == "success": return r        return rrandom.seed(42)h = MessageHistory()h.messages.append({"role":"user","content":"Find enterprise customers."})exec1 = NaiveRetryExecutor(h, max_retries=5)for q in ["enterprise","tier:gold","region:NA"]:    exec1.execute("search_db",{"q":q},0.7)print(f"NAIVE: {h.tool_count()} tool msgs, ~{h.token_estimate()} tokens")

## Wrong Fix: More RetriesIncreasing retry limit makes it worse -- more phantom results in context.

In [ ]:
random.seed(42)h2 = MessageHistory()h2.messages.append({"role":"user","content":"Find enterprise customers."})exec2 = NaiveRetryExecutor(h2, max_retries=10)for q in ["enterprise","tier:gold","region:NA"]:    exec2.execute("search_db",{"q":q},0.7)print(f"MORE RETRIES: {h2.tool_count()} tool msgs, ~{h2.token_estimate()} tokens")print(f"Token increase: {h2.token_estimate()/h.token_estimate():.1f}x worse")

## Correct Fix: Idempotency Key with Replace SemanticsEach logical tool call gets one call_id. Retries replace the previous result.

In [ ]:
class IdempotentExecutor:    def __init__(self, history, max_retries=3):        self.history = history        self.max_retries = max_retries    def execute(self, tool_name, args, fail_rate=0.7):        cid = f"{tool_name}_{hashlib.md5(json.dumps(args).encode()).hexdigest()[:8]}"        for attempt in range(self.max_retries):            if random.random() < fail_rate and attempt < self.max_retries - 1:                r = ToolResult(cid, tool_name, "error", error=f"Timeout attempt {attempt+1}")            else:                r = ToolResult(cid, tool_name, "success", data={"customers":[{"id":1}]})            replaced = False            for i, msg in enumerate(self.history.messages):                if msg.get("call_id") == cid:                    self.history.messages[i] = r.to_message()                    replaced = True                    break            if not replaced:                self.history.messages.append(r.to_message())            if r.status == "success": return r        return rrandom.seed(42)h3 = MessageHistory()h3.messages.append({"role":"user","content":"Find enterprise customers."})exec3 = IdempotentExecutor(h3, max_retries=5)for q in ["enterprise","tier:gold","region:NA"]:    exec3.execute("search_db",{"q":q},0.7)print(f"IDEMPOTENT: {h3.tool_count()} tool msgs, ~{h3.token_estimate()} tokens")print(f"\nComparison:")print(f"  Naive:      {h.token_estimate()} tokens, {h.tool_count()} tool msgs")print(f"  Idempotent: {h3.token_estimate()} tokens, {h3.tool_count()} tool msgs")print(f"  Savings:    {h.token_estimate()-h3.token_estimate()} tokens")

## Detection Signal`input_tokens` increasing >50% between sequential LLM calls in the same agent step.Retries in LLM agents are visible state, not transparent retry like an HTTP client.